# 06. Residual + Cell-Aware MMD Training

This notebook trains the stronger model variant that augments residual prediction with a cell-aware MMD penalty. The objective is to encourage the predicted latent distributions to match the true latent distributions within each cell type, thereby improving biological fidelity and OOD generalization. This notebook preserves the main ingredients from the current v2 training logic and saves the expanded training diagnostics needed for ablation analysis.


In [1]:
!pip -q install anndata scanpy scikit-learn scipy pandas numpy torch pyarrow

from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import urllib.request
from pathlib import Path

HELPER_DIR = Path("/content/drive/MyDrive/ChemDFM")
if str(HELPER_DIR) not in sys.path:
    sys.path.append(str(HELPER_DIR))

from chemdfm_notebook_helpers import *

DATA_PATH = Path("/content/drive/MyDrive/ChemDFM/data/raw/sciplex_complete_middle_subset.h5ad")
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    print("Downloading SciPlex dataset...")
    URL = "https://f003.backblazeb2.com/file/chemCPA-datasets/sciplex_complete_middle_subset.h5ad"
    urllib.request.urlretrieve(URL, DATA_PATH)
    print("Download complete.")

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_PATH =", DATA_PATH)
print("Using device:", DEVICE)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 104.8 MB/s eta 0:00:00
Mounted at /content/drive
PROJECT_ROOT = /content/drive/MyDrive/ChemDFM
DATA_PATH = /content/drive/MyDrive/ChemDFM/data/raw/sciplex_complete_middle_subset.h5ad
Using device: cuda


In [2]:
RUN_NAME = "residual_cellaware_mmd"
RUN_DIR = RUNS_DIR / RUN_NAME
CKPT_DIR = RUN_DIR / "checkpoints"
for p in [RUN_DIR, CKPT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

@dataclass
class CFG:
    split_col: str = "split_ho_pathway"
    pca_dim: int = 25
    batch_size: int = 512
    epochs: int = 12
    lr: float = 1e-3
    wd: float = 1e-5
    emb_dim: int = 32
    hidden: int = 256
    dose_hidden: int = 32
    alpha_topk: float = 2.0
    alpha_mmd: float = 0.05
    dropout: float = 0.1
    eval_topk: tuple = (20, 50)
    num_workers: int = 0
    pin_memory: bool = False
    k_top_train: int = 10
    min_count_mmd: int = 8
cfg = CFG()
save_json(asdict(cfg), RUN_DIR / "config_resolved.json")


In [4]:
adata, X = load_adata(split_col=cfg.split_col)
obs = pd.read_parquet(PROCESSED_DIR / "datasets" / "adata_obs_processed.parquet")
adata.obs = obs.copy()

X_pca = np.load(PROCESSED_DIR / "pca" / f"X_pca_{cfg.pca_dim}d.npy")
X0_pca = np.load(PROCESSED_DIR / "datasets" / f"X0_pca_{cfg.pca_dim}d.npy")
DELTA_pca = np.load(PROCESSED_DIR / "datasets" / f"DELTA_pca_{cfg.pca_dim}d.npy")

with open(PROCESSED_DIR / "encoders" / "drug_encoder.pkl", "rb") as f:
    drug_enc = pickle.load(f)
with open(PROCESSED_DIR / "encoders" / "cell_encoder.pkl", "rb") as f:
    cell_enc = pickle.load(f)

train_ds = ChemResidualDataset(adata, X_pca, X0_pca, DELTA_pca, split="train")
test_ds = ChemResidualDataset(adata, X_pca, X0_pca, DELTA_pca, split="test")
ood_ds = ChemResidualDataset(adata, X_pca, X0_pca, DELTA_pca, split="ood")

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)
ood_loader = DataLoader(ood_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)

model = ResidualDoseResponseModel(
    latent_dim=cfg.pca_dim,
    n_drugs=len(drug_enc.classes_),
    n_cells=len(cell_enc.classes_),
    emb_dim=cfg.emb_dim,
    hidden=cfg.hidden,
    dose_hidden=cfg.dose_hidden,
    dropout=cfg.dropout,
).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.wd)
history, best_score = [], -1e9
best_path = CKPT_DIR / "best_residual_cellaware_mmd.pt"


/usr/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [5]:
for epoch in range(1, cfg.epochs + 1):
    model.train()
    train_losses, train_mse, train_topk, train_mmd = [], [], [], []

    for batch in train_loader:
        x0 = batch["x0"].to(DEVICE)
        x_true = batch["x_true"].to(DEVICE)
        delta_true = batch["delta"].to(DEVICE)
        drug_idx = batch["drug_idx"].to(DEVICE)
        cell_idx = batch["cell_idx"].to(DEVICE)
        dose = batch["dose"].to(DEVICE)

        optimizer.zero_grad()
        delta_hat, x_hat = model(x0, drug_idx, cell_idx, dose)

        loss_mse = F.mse_loss(delta_hat, delta_true)
        mask_topk = topk_mask_from_true(delta_true, k=min(cfg.k_top_train, cfg.pca_dim))
        loss_topk = ((delta_hat - delta_true) ** 2 * mask_topk).sum() / (mask_topk.sum() + 1e-8)
        loss_mmd = cell_aware_mmd(x_hat, x_true, cell_idx, min_count=cfg.min_count_mmd)
        loss = loss_mse + cfg.alpha_topk * loss_topk + cfg.alpha_mmd * loss_mmd

        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        train_mse.append(loss_mse.item())
        train_topk.append(loss_topk.item())
        train_mmd.append(loss_mmd.item())

    test_metrics, test_per_cell, test_group, _, _, _ = evaluate_loader(model, test_loader, eval_topk=cfg.eval_topk, split_name="test")
    ood_metrics, ood_per_cell, ood_group, _, _, _ = evaluate_loader(model, ood_loader, eval_topk=cfg.eval_topk, split_name="ood")

    def pick_cell(df, cell, key):
        sub = df[df["cell_type"] == cell]
        return float(sub.iloc[0][key]) if len(sub) else np.nan

    k562_test_top50 = pick_cell(test_per_cell, "K562", "r2_top50")
    k562_ood_top50 = pick_cell(ood_per_cell, "K562", "r2_top50")

    score = (
        0.50 * test_metrics["r2_top50"] +
        0.25 * ood_metrics["r2_top50"] +
        0.15 * (0.0 if np.isnan(k562_test_top50) else k562_test_top50) +
        0.10 * (0.0 if np.isnan(k562_ood_top50) else k562_ood_top50)
    )

    row = {
        "epoch": epoch,
        "train_loss": float(np.mean(train_losses)),
        "train_mse": float(np.mean(train_mse)),
        "train_topk": float(np.mean(train_topk)),
        "train_mmd": float(np.mean(train_mmd)),
        "test_r2_top50": test_metrics["r2_top50"],
        "ood_r2_top50": ood_metrics["r2_top50"],
        "test_r2_full": test_metrics["r2_full"],
        "ood_r2_full": ood_metrics["r2_full"],
        "k562_test_r2_top50": k562_test_top50,
        "k562_ood_r2_top50": k562_ood_top50,
        "selection_score": float(score),
    }
    history.append(row)
    pd.DataFrame(history).to_csv(RUN_DIR / "training_history.csv", index=False)

    print(
        f"Epoch {epoch:02d} | TrainLoss {row['train_loss']:.4f} | "
        f"Test top50 {row['test_r2_top50']:.4f} | OOD top50 {row['ood_r2_top50']:.4f} | "
        f"K562 test {row['k562_test_r2_top50']:.4f} | K562 OOD {row['k562_ood_r2_top50']:.4f}"
    )

    if score > best_score:
        best_score = score
        torch.save(model.state_dict(), best_path)
        test_per_cell.to_csv(RUN_DIR / "best_test_per_cell.csv", index=False)
        ood_per_cell.to_csv(RUN_DIR / "best_ood_per_cell.csv", index=False)
        test_group.to_csv(RUN_DIR / "best_test_groupwise.csv", index=False)
        ood_group.to_csv(RUN_DIR / "best_ood_groupwise.csv", index=False)
        print("  ✅ saved best model")


Epoch 01 | TrainLoss 1.8894 | Test top50 0.5749 | OOD top50 0.5484 | K562 test 0.5853 | K562 OOD 0.5609
  ✅ saved best model
Epoch 02 | TrainLoss 1.8390 | Test top50 0.5758 | OOD top50 0.5475 | K562 test 0.5865 | K562 OOD 0.5563
Epoch 03 | TrainLoss 1.8285 | Test top50 0.5758 | OOD top50 0.5539 | K562 test 0.5864 | K562 OOD 0.5652
  ✅ saved best model
Epoch 04 | TrainLoss 1.8239 | Test top50 0.5764 | OOD top50 0.5488 | K562 test 0.5869 | K562 OOD 0.5592
Epoch 05 | TrainLoss 1.8204 | Test top50 0.5754 | OOD top50 0.5477 | K562 test 0.5849 | K562 OOD 0.5553
Epoch 06 | TrainLoss 1.8181 | Test top50 0.5756 | OOD top50 0.5497 | K562 test 0.5847 | K562 OOD 0.5577
Epoch 07 | TrainLoss 1.8167 | Test top50 0.5758 | OOD top50 0.5566 | K562 test 0.5846 | K562 OOD 0.5574
Epoch 08 | TrainLoss 1.8152 | Test top50 0.5761 | OOD top50 0.5545 | K562 test 0.5853 | K562 OOD 0.5485
Epoch 09 | TrainLoss 1.8143 | Test top50 0.5762 | OOD top50 0.5522 | K562 test 0.5848 | K562 OOD 0.5537
Epoch 10 | TrainLoss 1

In [6]:
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
final_test_metrics, final_test_per_cell, _, pred_test, true_test, x0_test = evaluate_loader(model, test_loader, eval_topk=cfg.eval_topk, split_name="test")
final_ood_metrics, final_ood_per_cell, _, pred_ood, true_ood, x0_ood = evaluate_loader(model, ood_loader, eval_topk=cfg.eval_topk, split_name="ood")

final_overall = pd.DataFrame([
    {"split": "test", "model": "ResidualDoseResponseModel_CellAwareMMD", **final_test_metrics},
    {"split": "ood", "model": "ResidualDoseResponseModel_CellAwareMMD", **final_ood_metrics},
])
final_overall.to_csv(RUN_DIR / "final_overall_metrics.csv", index=False)
pd.concat([final_test_per_cell, final_ood_per_cell], ignore_index=True).to_csv(RUN_DIR / "final_per_cell_metrics.csv", index=False)
np.save(RUN_DIR / "pred_test.npy", pred_test)
np.save(RUN_DIR / "pred_ood.npy", pred_ood)

final_overall


,split,model,r2_full,pearson_rowmean,mse,collapse_ratio,mean_shift_error,r2_top20,r2_top50
0,test,ResidualDoseResponseModel_CellAwareMMD,0.635105,0.783744,0.342062,0.647114,0.442125,0.498540,0.575434
1,ood,ResidualDoseResponseModel_CellAwareMMD,0.615382,0.772938,0.358423,0.596353,0.453687,0.486709,0.558485


This notebook produces the stronger canonical model run. The next notebook reuses its saved checkpoint and the shared PCA model to reconstruct gene-space predictions and compute biological diagnostics from disk.
